# 3D U-Net for Lung Nodule Detection in CT Scans
### Deep Learning with PyTorch — Medical Imaging Project

**Author:** Aidan Agee  
**Dataset:** [LUNA16](https://luna16.grand-challenge.org/) — Lung Nodule Analysis 2016  
**Framework:** PyTorch  
**Task:** Binary segmentation of lung nodules in 3D CT volumes

---

## Overview

This project implements a **3D U-Net** architecture for automated detection and segmentation of pulmonary nodules in chest CT scans. Early detection of lung nodules is critical for identifying potential malignancies — this model learns to segment nodule regions directly from volumetric imaging data.

This work extends my earlier project on [intracranial aneurysm detection](https://github.com/aidanagee/aneurysm-detection-rsna), applying similar 3D segmentation techniques to a different anatomical domain.

### Key Concepts Covered
- 3D convolutional neural networks for volumetric medical imaging
- U-Net encoder-decoder architecture with skip connections
- DICOM data preprocessing and normalization
- Handling severe class imbalance in medical segmentation tasks
- Dice loss and IoU evaluation metrics
- Data augmentation strategies for 3D volumes


---
## 1. Architecture: Why U-Net for Medical Segmentation?

The **U-Net** architecture was originally proposed for 2D biomedical image segmentation (Ronneberger et al., 2015) and has since become the dominant architecture for medical imaging tasks. We extend it to 3D to fully exploit the volumetric nature of CT data.

### Why U-Net works so well here:
- **Skip connections** preserve fine spatial detail lost during downsampling — critical for precise nodule boundary detection
- **Encoder-decoder structure** captures both global context (what region of the lung) and local detail (exact nodule boundary)
- **Works well with limited data** — medical datasets are small; U-Net's architecture is inherently data-efficient

### 2D vs 3D U-Net:
| Feature | 2D U-Net | 3D U-Net |
|---|---|---|
| Input | Single slice (H x W) | Full volume (D x H x W) |
| Convolutions | 2D conv | 3D conv |
| Context | Within-slice only | Full volumetric context |
| Memory cost | Low | High |
| Nodule detection | Misses inter-slice patterns | Captures full 3D shape |

For lung nodules — which are roughly spherical 3D structures — processing the full volume is significantly more effective than processing individual slices.


---
## 2. Imports and Environment Setup

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
import random

# For DICOM/medical imaging
try:
    import SimpleITK as sitk
    print("SimpleITK available")
except ImportError:
    print("SimpleITK not installed — run: pip install SimpleITK")

try:
    import pandas as pd
    print("Pandas available")
except ImportError:
    print("Pandas not installed — run: pip install pandas")

# Reproducibility — always set seeds in ML projects
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## 3. Data Preprocessing

### The LUNA16 Dataset
LUNA16 provides 888 CT scans with annotations for 1,186 nodules across 10 subsets. Each CT scan:
- Is stored in MHD/RAW format (MetaImage — similar to DICOM)
- Has voxel spacing that varies between scans (must be resampled to uniform spacing)
- Contains a candidates CSV with nodule coordinates and labels

### Critical Preprocessing Steps

#### 3.1 HU Clipping
CT values are measured in **Hounsfield Units (HU)**:
- Air: ~-1000 HU
- Lung tissue: ~-500 HU  
- Soft tissue: ~40 HU
- Bone: ~400+ HU

We clip to **[-1000, 400]** to remove irrelevant extreme values, then normalize to [0, 1].

#### 3.2 Resampling to Isotropic Spacing
CT scanners produce voxels with non-uniform spacing (e.g., 0.7mm x 0.7mm x 2.5mm). 
We resample all volumes to **1mm x 1mm x 1mm** so the network sees consistent spatial relationships.

#### 3.3 Patch Extraction
Processing a full CT volume (~512x512x300+) is too memory-intensive. We extract fixed-size **64x64x64** patches centered on candidate nodule locations.


In [ ]:
def load_ct_volume(mhd_path):
    """
    Load a CT volume from MHD/RAW format using SimpleITK.
    Returns the numpy array and the SimpleITK image object (for spacing info).
    """
    itk_image = sitk.ReadImage(str(mhd_path))
    ct_array = sitk.GetArrayFromImage(itk_image)  # shape: (Z, Y, X)
    return ct_array, itk_image


def resample_to_isotropic(itk_image, new_spacing=(1.0, 1.0, 1.0)):
    """
    Resample CT volume to isotropic 1mm spacing.
    
    Why this matters: A nodule that's 3mm across should look the same size
    regardless of which scanner captured it. Without resampling, the same 
    physical nodule would appear at different scales across different scans.
    """
    original_spacing = itk_image.GetSpacing()  # (x, y, z) in mm
    original_size = itk_image.GetSize()        # (x, y, z) in voxels
    
    # Calculate new size to maintain physical dimensions
    new_size = [
        int(round(orig_sz * orig_sp / new_sp))
        for orig_sz, orig_sp, new_sp 
        in zip(original_size, original_spacing, new_spacing)
    ]
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetOutputDirection(itk_image.GetDirection())
    resampler.SetOutputOrigin(itk_image.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(-1000)  # fill with air HU value
    
    resampled = resampler.Execute(itk_image)
    return sitk.GetArrayFromImage(resampled)


def normalize_hu(ct_array, min_hu=-1000, max_hu=400):
    """
    Clip HU values to lung-relevant range and normalize to [0, 1].
    
    Clipping removes bone and extreme air values that aren't useful
    for distinguishing lung nodules from surrounding tissue.
    """
    ct_clipped = np.clip(ct_array, min_hu, max_hu)
    ct_normalized = (ct_clipped - min_hu) / (max_hu - min_hu)
    return ct_normalized.astype(np.float32)


def world_to_voxel(coord_world, origin, spacing):
    """
    Convert real-world mm coordinates (from annotations CSV) 
    to voxel indices in the CT array.
    
    CT annotations are stored in world coordinates (mm from scanner origin).
    We need voxel indices to actually extract patches.
    """
    return np.round((coord_world - origin) / spacing).astype(int)


def extract_patch(ct_volume, center_voxel, patch_size=(64, 64, 64)):
    """
    Extract a 3D patch centered on a candidate nodule location.
    Pads with zeros if patch extends beyond volume boundary.
    """
    pz, py, px = patch_size
    cz, cy, cx = center_voxel
    
    # Calculate patch boundaries
    z0, z1 = cz - pz//2, cz + pz//2
    y0, y1 = cy - py//2, cy + py//2
    x0, x1 = cx - px//2, cx + px//2
    
    # Handle boundary cases with padding
    padded = np.zeros(patch_size, dtype=np.float32)
    
    # Clamp to volume dimensions
    sz, sy, sx = ct_volume.shape
    
    vz0, vz1 = max(0, z0), min(sz, z1)
    vy0, vy1 = max(0, y0), min(sy, y1)
    vx0, vx1 = max(0, x0), min(sx, x1)
    
    pz0 = vz0 - z0
    py0 = vy0 - y0
    px0 = vx0 - x0
    
    padded[pz0:pz0+(vz1-vz0), py0:py0+(vy1-vy0), px0:px0+(vx1-vx0)] =         ct_volume[vz0:vz1, vy0:vy1, vx0:vx1]
    
    return padded


# ── Visualize preprocessing steps ──────────────────────────────────────────
def visualize_preprocessing_demo():
    """
    Demonstrate preprocessing on a synthetic volume.
    In practice, replace with actual LUNA16 data.
    """
    # Simulate a CT patch with a nodule (bright sphere in lung-like background)
    synthetic_ct = np.random.normal(-500, 100, (64, 64, 64)).astype(np.float32)
    
    # Add a synthetic nodule (higher density sphere)
    cz, cy, cx = 32, 32, 32
    radius = 6
    z, y, x = np.ogrid[:64, :64, :64]
    mask = (z-cz)**2 + (y-cy)**2 + (x-cx)**2 <= radius**2
    synthetic_ct[mask] = np.random.normal(40, 20, mask.sum())
    
    normalized = normalize_hu(synthetic_ct)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].imshow(synthetic_ct[32], cmap='gray')
    axes[0].set_title('Raw HU Values\n(center axial slice)')
    axes[0].set_xlabel('HU range: approx -1000 to 400')
    
    axes[1].imshow(normalized[32], cmap='gray')
    axes[1].set_title('After HU Normalization\n[0, 1] range')
    axes[1].set_xlabel('Nodule visible as brighter region')
    
    axes[2].imshow(normalized[32], cmap='hot')
    axes[2].set_title('Heat Map View\nHighlights high-density regions')
    axes[2].set_xlabel('Red = higher density (nodule candidate)')
    
    plt.suptitle('Preprocessing Pipeline — Synthetic CT Patch Demo', fontsize=13)
    plt.tight_layout()
    plt.savefig('preprocessing_demo.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: preprocessing_demo.png")

visualize_preprocessing_demo()

---
## 4. PyTorch Dataset Class

The `LunaNoduleDataset` class handles:
- Loading preprocessed CT patches from disk
- Generating binary segmentation masks (sphere at annotated nodule location)
- 3D data augmentation to improve generalization
- Balancing positive (nodule) vs negative (non-nodule) samples

### Class Imbalance Problem
This is one of the most critical challenges in medical segmentation. In a typical CT scan:
- ~99.9% of voxels are **not** nodule
- ~0.1% of voxels **are** nodule

If we don't address this, the model learns to predict "no nodule everywhere" and achieves 99.9% accuracy while being completely useless. We handle this by:
1. **Balanced sampling** — equal positive/negative patches during training
2. **Dice loss** — a loss function that's robust to class imbalance


In [ ]:
class LunaNoduleDataset(Dataset):
    """
    PyTorch Dataset for LUNA16 lung nodule segmentation.
    
    Expected directory structure:
        data/
            preprocessed/
                subset0/
                    <seriesuid>_<x>_<y>_<z>.npy   # CT patch
                    <seriesuid>_<x>_<y>_<z>_mask.npy  # segmentation mask
            candidates_V2.csv
    
    For demonstration, we use synthetic data if real data isn't available.
    """
    
    def __init__(self, data_dir, split='train', patch_size=64, 
                 augment=True, use_synthetic=True):
        self.data_dir = Path(data_dir)
        self.split = split
        self.patch_size = patch_size
        self.augment = augment and (split == 'train')
        self.use_synthetic = use_synthetic
        
        if use_synthetic:
            self.samples = self._generate_synthetic_samples(n=200)
            print(f"Using synthetic data: {len(self.samples)} samples ({split})")
        else:
            self.samples = self._load_real_samples()
            print(f"Loaded {len(self.samples)} real samples ({split})")
    
    def _generate_synthetic_samples(self, n=200):
        """Generate synthetic CT patches for demonstration/testing."""
        samples = []
        n_positive = n // 2  # 50/50 split for demonstration
        
        for i in range(n):
            is_positive = i < n_positive
            samples.append({
                'id': f'synthetic_{i:04d}',
                'is_nodule': is_positive,
                'nodule_radius': np.random.uniform(3, 10) if is_positive else 0
            })
        
        # Split train/val
        split_idx = int(0.8 * len(samples))
        if self.split == 'train':
            return samples[:split_idx]
        else:
            return samples[split_idx:]
    
    def _make_synthetic_patch(self, sample):
        """Create a synthetic CT patch with optional nodule."""
        ps = self.patch_size
        
        # Background: lung-like tissue with noise
        patch = np.random.normal(-500, 80, (ps, ps, ps)).astype(np.float32)
        mask = np.zeros((ps, ps, ps), dtype=np.float32)
        
        if sample['is_nodule']:
            # Place a spherical nodule at center with some jitter
            r = sample['nodule_radius']
            jitter = np.random.randint(-8, 8, 3)
            cz = ps//2 + jitter[0]
            cy = ps//2 + jitter[1]  
            cx = ps//2 + jitter[2]
            
            z, y, x = np.ogrid[:ps, :ps, :ps]
            sphere = (z-cz)**2 + (y-cy)**2 + (x-cx)**2 <= r**2
            
            # Nodule has higher HU than surrounding tissue
            patch[sphere] = np.random.normal(40, 30, sphere.sum())
            mask[sphere] = 1.0
        
        # Normalize to [0, 1]
        patch = normalize_hu(patch)
        return patch, mask
    
    def augment_3d(self, patch, mask):
        """
        Apply random 3D augmentations.
        
        Augmentation is essential with small medical datasets.
        We apply simple flips and rotations that preserve nodule appearance.
        """
        # Random flips along each axis
        for axis in range(3):
            if random.random() > 0.5:
                patch = np.flip(patch, axis=axis).copy()
                mask = np.flip(mask, axis=axis).copy()
        
        # Random 90-degree rotations in axial plane
        k = random.randint(0, 3)
        if k > 0:
            patch = np.rot90(patch, k=k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k=k, axes=(1, 2)).copy()
        
        # Random intensity scaling (simulates scanner variability)
        if random.random() > 0.5:
            scale = random.uniform(0.9, 1.1)
            patch = np.clip(patch * scale, 0, 1)
        
        return patch, mask
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        if self.use_synthetic:
            patch, mask = self._make_synthetic_patch(sample)
        else:
            patch = np.load(self.data_dir / 'preprocessed' / f"{sample['id']}.npy")
            mask = np.load(self.data_dir / 'preprocessed' / f"{sample['id']}_mask.npy")
        
        if self.augment:
            patch, mask = self.augment_3d(patch, mask)
        
        # Add channel dimension: (1, D, H, W)
        patch_tensor = torch.from_numpy(patch).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)
        
        return patch_tensor, mask_tensor


# Test dataset
train_dataset = LunaNoduleDataset('data/', split='train', use_synthetic=True)
val_dataset = LunaNoduleDataset('data/', split='val', use_synthetic=True)

patch, mask = train_dataset[0]
print(f"Patch shape: {patch.shape}  dtype: {patch.dtype}")
print(f"Mask shape:  {mask.shape}   dtype: {mask.dtype}")
print(f"Patch value range: [{patch.min():.3f}, {patch.max():.3f}]")
print(f"Mask unique values: {torch.unique(mask)}")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)
print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

---
## 5. 3D U-Net Architecture

The U-Net has two paths:

**Encoder (contracting path):** Progressively downsamples the volume using 3D convolutions and max pooling, building a hierarchy of features from low-level (edges, textures) to high-level (shape, context).

**Decoder (expanding path):** Upsamples back to original resolution using transposed convolutions, combining with encoder features via skip connections.

**Skip connections:** Concatenate encoder feature maps directly to the corresponding decoder level. This is the key innovation — the decoder has access to both the global context from the bottleneck AND the fine spatial detail from the encoder. Without skip connections, precise boundary delineation is impossible.

```
Input (1, 64, 64, 64)
    │
    ▼
[Encoder Block 1] ──────────────────────────────────► [Decoder Block 4]
    │ MaxPool                                              ▲ Concat + Conv
    ▼                                                      │
[Encoder Block 2] ──────────────────────────────► [Decoder Block 3]
    │ MaxPool                                          ▲ Concat + Conv
    ▼                                                  │
[Encoder Block 3] ──────────────────────────► [Decoder Block 2]
    │ MaxPool                                      ▲ Concat + Conv
    ▼                                              │
[Encoder Block 4] ──────────────────► [Decoder Block 1]
    │ MaxPool                              ▲ Concat + Conv
    ▼                                      │
[Bottleneck] ──────────────────────────────
    
Output (1, 64, 64, 64) — sigmoid activation → segmentation mask
```


In [ ]:
class DoubleConv3D(nn.Module):
    """
    Two consecutive 3D convolution + BatchNorm + ReLU blocks.
    This is the fundamental building block of U-Net.
    
    BatchNorm is critical for training stability — normalizes activations
    to prevent vanishing/exploding gradients in deep networks.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class EncoderBlock(nn.Module):
    """Encoder block: MaxPool downsampling followed by DoubleConv."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.conv = DoubleConv3D(in_channels, out_channels)
    
    def forward(self, x):
        return self.conv(self.pool(x))


class DecoderBlock(nn.Module):
    """
    Decoder block: Transposed conv upsampling + skip connection concatenation + DoubleConv.
    
    The skip connection doubles the channel count (hence in_channels * 2 in DoubleConv).
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Upsample and halve channels
        self.upsample = nn.ConvTranspose3d(in_channels, in_channels // 2, 
                                            kernel_size=2, stride=2)
        # After concat with skip: in_channels // 2 + in_channels // 2 = in_channels
        self.conv = DoubleConv3D(in_channels, out_channels)
    
    def forward(self, x, skip):
        x = self.upsample(x)
        
        # Handle size mismatch from odd input dimensions
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=True)
        
        # Concatenate along channel dimension
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNet3D(nn.Module):
    """
    Full 3D U-Net for volumetric medical image segmentation.
    
    Architecture follows the original U-Net paper adapted for 3D:
    - Ronneberger et al. (2015) — original 2D U-Net
    - Cicek et al. (2016) — 3D U-Net extension
    
    Feature channels: 32 → 64 → 128 → 256 → 512 (bottleneck)
    We use 32 as base (vs original 64) to fit in GPU memory for 3D volumes.
    """
    
    def __init__(self, in_channels=1, out_channels=1, base_features=32):
        super().__init__()
        f = base_features  # 32
        
        # Encoder
        self.enc1 = DoubleConv3D(in_channels, f)       # 1 → 32
        self.enc2 = EncoderBlock(f, f * 2)              # 32 → 64
        self.enc3 = EncoderBlock(f * 2, f * 4)          # 64 → 128
        self.enc4 = EncoderBlock(f * 4, f * 8)          # 128 → 256
        
        # Bottleneck
        self.bottleneck = EncoderBlock(f * 8, f * 16)   # 256 → 512
        
        # Decoder
        self.dec4 = DecoderBlock(f * 16, f * 8)         # 512 → 256
        self.dec3 = DecoderBlock(f * 8, f * 4)          # 256 → 128
        self.dec2 = DecoderBlock(f * 4, f * 2)          # 128 → 64
        self.dec1 = DecoderBlock(f * 2, f)              # 64 → 32
        
        # Final 1x1 conv to produce segmentation map
        self.final_conv = nn.Conv3d(f, out_channels, kernel_size=1)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """He initialization for conv layers — better than default for ReLU networks."""
        for m in self.modules():
            if isinstance(m, nn.Conv3d) or isinstance(m, nn.ConvTranspose3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x):
        # Encoder — save outputs for skip connections
        s1 = self.enc1(x)       # (B, 32, 64, 64, 64)
        s2 = self.enc2(s1)      # (B, 64, 32, 32, 32)
        s3 = self.enc3(s2)      # (B, 128, 16, 16, 16)
        s4 = self.enc4(s3)      # (B, 256, 8, 8, 8)
        
        # Bottleneck
        b = self.bottleneck(s4) # (B, 512, 4, 4, 4)
        
        # Decoder — pass skip connections
        d4 = self.dec4(b, s4)   # (B, 256, 8, 8, 8)
        d3 = self.dec3(d4, s3)  # (B, 128, 16, 16, 16)
        d2 = self.dec2(d3, s2)  # (B, 64, 32, 32, 32)
        d1 = self.dec1(d2, s1)  # (B, 32, 64, 64, 64)
        
        # Final segmentation map (no sigmoid — handled in loss function)
        return self.final_conv(d1)  # (B, 1, 64, 64, 64)
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Instantiate and inspect model
model = UNet3D(in_channels=1, out_channels=1, base_features=32).to(device)
print(f"Total trainable parameters: {model.count_parameters():,}")

# Test forward pass with a batch
x = torch.randn(2, 1, 64, 64, 64).to(device)
with torch.no_grad():
    out = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Output range: [{out.min().item():.3f}, {out.max().item():.3f}] (pre-sigmoid logits)")

---
## 6. Loss Function: Dice Loss + Binary Cross Entropy

For medical segmentation with severe class imbalance, standard BCE loss fails — the model learns to predict all zeros. We use a combined loss:

**Dice Loss** directly optimizes the Dice coefficient (the metric we care about). It's naturally robust to class imbalance because it measures overlap regardless of absolute counts.

$$\text{Dice} = \frac{2 |P \cap G|}{|P| + |G|}$$

**Combined Loss = Dice Loss + BCE Loss**
BCE provides pixel-level gradients (good for learning boundaries), Dice provides region-level signal (good for handling imbalance). Together they train more stably than either alone.


In [ ]:
class DiceLoss(nn.Module):
    """
    Soft Dice Loss for binary segmentation.
    
    'Soft' means we use raw probabilities (after sigmoid) rather than 
    hard 0/1 predictions, making it differentiable for backpropagation.
    
    smooth=1.0 prevents division by zero and acts as Laplace smoothing.
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        
        # Flatten spatial dimensions for calculation
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (probs_flat * targets_flat).sum()
        dice_coeff = (2. * intersection + self.smooth) /                      (probs_flat.sum() + targets_flat.sum() + self.smooth)
        
        return 1 - dice_coeff  # Loss = 1 - Dice (minimize loss = maximize Dice)


class CombinedLoss(nn.Module):
    """Weighted combination of Dice Loss and Binary Cross Entropy."""
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCEWithLogitsLoss()  # numerically stable BCE
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
    
    def forward(self, logits, targets):
        dice = self.dice_loss(logits, targets)
        bce = self.bce_loss(logits, targets)
        return self.dice_weight * dice + self.bce_weight * bce


def dice_coefficient(preds, targets, threshold=0.5, smooth=1.0):
    """
    Evaluation metric — compute Dice coefficient on binary predictions.
    Different from DiceLoss: uses hard threshold, not soft probabilities.
    """
    preds_binary = (torch.sigmoid(preds) > threshold).float()
    
    preds_flat = preds_binary.view(-1)
    targets_flat = targets.view(-1)
    
    intersection = (preds_flat * targets_flat).sum()
    return ((2. * intersection + smooth) / 
            (preds_flat.sum() + targets_flat.sum() + smooth)).item()


# Test loss functions
criterion = CombinedLoss(dice_weight=0.5, bce_weight=0.5)
dummy_pred = torch.randn(2, 1, 64, 64, 64)
dummy_mask = torch.zeros(2, 1, 64, 64, 64)
dummy_mask[:, :, 28:36, 28:36, 28:36] = 1.0  # simulate nodule region

loss = criterion(dummy_pred, dummy_mask)
dice = dice_coefficient(dummy_pred, dummy_mask)
print(f"Combined loss (random predictions): {loss.item():.4f}")
print(f"Dice coefficient (random predictions): {dice:.4f}  (expect ~0 for random)")

---
## 7. Training Loop

Key training decisions:
- **Optimizer:** Adam with lr=1e-4 — adaptive learning rate works well for medical imaging
- **LR Scheduler:** ReduceLROnPlateau — halves learning rate when val loss plateaus
- **Early stopping:** Prevents overfitting on small medical datasets
- **Gradient clipping:** Prevents exploding gradients with 3D convolutions


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, grad_clip=1.0):
    """Run one training epoch. Returns avg loss and avg Dice."""
    model.train()
    total_loss, total_dice = 0.0, 0.0
    
    for batch_idx, (patches, masks) in enumerate(loader):
        patches = patches.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        logits = model(patches)
        loss = criterion(logits, masks)
        
        loss.backward()
        
        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice_coefficient(logits.detach(), masks)
    
    n = len(loader)
    return total_loss / n, total_dice / n


def validate(model, loader, criterion, device):
    """Run validation. Returns avg loss and avg Dice."""
    model.eval()
    total_loss, total_dice = 0.0, 0.0
    
    with torch.no_grad():
        for patches, masks in loader:
            patches = patches.to(device)
            masks = masks.to(device)
            
            logits = model(patches)
            loss = criterion(logits, masks)
            
            total_loss += loss.item()
            total_dice += dice_coefficient(logits, masks)
    
    n = len(loader)
    return total_loss / n, total_dice / n


def train_model(model, train_loader, val_loader, n_epochs=20, lr=1e-4, patience=5):
    """
    Full training loop with LR scheduling and early stopping.
    
    Args:
        patience: Stop training if val loss doesn't improve for this many epochs
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    criterion = CombinedLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Train Dice':>12} {'Val Dice':>10} {'LR':>10}")
    print("-" * 65)
    
    for epoch in range(1, n_epochs + 1):
        train_loss, train_dice = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_dice = validate(model, val_loader, criterion, device)
        
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        
        print(f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} "
              f"{train_dice:>12.4f} {val_dice:>10.4f} {current_lr:>10.2e}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break
    
    # Restore best weights
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"\nRestored best model (val loss: {best_val_loss:.4f})")
    
    return history


# Train the model
print("Starting training...\n")
history = train_model(
    model, train_loader, val_loader,
    n_epochs=15,
    lr=1e-4,
    patience=5
)

---
## 8. Results & Visualization

In [ ]:
def plot_training_history(history):
    """Plot loss and Dice curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
    axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val Loss', markersize=4)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Combined Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs, history['train_dice'], 'b-o', label='Train Dice', markersize=4)
    axes[1].plot(epochs, history['val_dice'], 'r-o', label='Val Dice', markersize=4)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Dice Coefficient')
    axes[1].set_title('Training & Validation Dice')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('3D U-Net Training Progress — Lung Nodule Detection', fontsize=13)
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: training_curves.png")


def visualize_predictions(model, dataset, n_samples=3, threshold=0.5):
    """
    Visualize model predictions vs ground truth on sample patches.
    Shows center axial slice of each 3D patch.
    """
    model.eval()
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    
    with torch.no_grad():
        for i in range(n_samples):
            patch, mask = dataset[i]
            logit = model(patch.unsqueeze(0).to(device))
            pred = (torch.sigmoid(logit) > threshold).float()
            
            # Get center slice
            center = patch.shape[-1] // 2
            ct_slice = patch[0, center].cpu().numpy()
            mask_slice = mask[0, center].cpu().numpy()
            pred_slice = pred[0, 0, center].cpu().numpy()
            
            axes[i, 0].imshow(ct_slice, cmap='gray')
            axes[i, 0].set_title(f'CT Patch (axial slice {center})')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(ct_slice, cmap='gray')
            axes[i, 1].imshow(mask_slice, cmap='Reds', alpha=0.5)
            axes[i, 1].set_title('Ground Truth Mask')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(ct_slice, cmap='gray')
            axes[i, 2].imshow(pred_slice, cmap='Blues', alpha=0.5)
            d = dice_coefficient(logit.cpu(), mask.unsqueeze(0))
            axes[i, 2].set_title(f'Prediction (Dice={d:.3f})')
            axes[i, 2].axis('off')
    
    plt.suptitle('3D U-Net Predictions — Lung Nodule Segmentation', fontsize=13)
    plt.tight_layout()
    plt.savefig('predictions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: predictions.png")


plot_training_history(history)
visualize_predictions(model, val_dataset)

---
## 9. Discussion & Limitations

### Results on Synthetic Data
This notebook demonstrates the full pipeline using synthetic CT patches. The model successfully learns to detect synthetic nodule-like structures, validating the architecture and training setup.

### Expected Performance on Real LUNA16 Data
State-of-the-art 3D U-Net models on LUNA16 achieve:
- **Sensitivity:** ~90-95% (fraction of real nodules detected)
- **False positive rate:** ~1-4 per scan
- **Dice coefficient:** ~0.6-0.8 on nodule voxels

### Key Challenges with Real Data
1. **Small nodule size** — nodules can be as small as 3mm, covering very few voxels in a 64³ patch
2. **Severe class imbalance** — even within a positive patch, nodule voxels are <5% of total
3. **Annotation variability** — radiologists disagree on borderline cases (~20% of nodules)
4. **Scanner variability** — different CT protocols produce different HU distributions

### Similarities to Prior Work
This project extends the approach from my [RSNA 2024 aneurysm detection project](https://github.com/aidanagee/aneurysm-detection-rsna):
- Both use 3D U-Net for volumetric segmentation
- Both address class imbalance with Dice loss
- Both process DICOM/medical imaging formats
- Key difference: lung nodules are more spherical and less complex in shape than aneurysms

### To Run on Real LUNA16 Data
1. Download LUNA16 from https://luna16.grand-challenge.org/
2. Run the preprocessing functions in Section 3 on each scan
3. Update `LunaNoduleDataset` to load from your preprocessed directory
4. Training on GPU recommended (~2-4 hours for full dataset)


---
## References

1. Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation. *MICCAI 2015.*
2. Çiçek, Ö., et al. (2016). 3D U-Net: Learning Dense Volumetric Segmentation from Sparse Annotation. *MICCAI 2016.*
3. Setio, A.A.A., et al. (2017). Validation, comparison, and combination of algorithms for automatic detection of pulmonary nodules in computed tomography images. *Medical Image Analysis.*
4. Stevens, E., Antiga, L., & Viehmann, T. (2020). *Deep Learning with PyTorch.* Manning Publications.
